# Steps analysis: from a recorded row to a defensible conclusion
Prepared for the internship presentation on **Friday, October 23, 2026**.

**Question:** What does a steps record represent, how reliable are its keys and measurement periods, and how often do recorded steps meet a valid recorded goal?

Run cells in order. Only `steps`, `activity`, and `goals` are read; the database is never modified. Outputs contain aggregate results, not participant identifiers or raw records. Equal table lengths are not evidence of matching records. This is descriptive analysis, not a causal or health assessment.

## 1. Load the three necessary tables
The connection is read privately from the existing `.env`. We inspect SQL types as well as pandas types. A timestamp suffix `_sec` suggests seconds, but timezone and interval conventions still require source documentation.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sqlalchemy import create_engine, inspect, text
from dotenv import dotenv_values
from IPython.display import display

PROJECT = Path.cwd()
assert (PROJECT / '.env').exists(), 'Run this notebook from the project directory.'
url = dotenv_values(PROJECT / '.env').get('DATABASE_URL')
assert url, 'DATABASE_URL must be configured privately in .env.'
engine = create_engine(url)
schema_rows = []
tables = {}
try:
    inspector = inspect(engine)
    for name in ['steps', 'activity', 'goals']:
        for c in inspector.get_columns(name):
            schema_rows.append({'table': name, 'column': c['name'], 'sql_type': str(c['type'])})
        with engine.connect() as connection:
            tables[name] = pd.read_sql(text('SELECT * FROM ' + name), connection)
finally:
    engine.dispose()
del url
display(pd.DataFrame(schema_rows))
display(pd.DataFrame([{'table': k, 'rows': len(v), 'columns': v.shape[1],
                      'participants': v.sdv_user_id.nunique()} for k,v in tables.items()]))
s = tables['steps'].copy()
for frame in tables.values():
    frame['calendar_date'] = pd.to_datetime(frame.calendar_date, errors='coerce')
s = tables['steps'].copy()


## 2. Understand one row without revealing a real participant
A **synthetic example** is: participant A, calendar date 2026-01-05, recorded period 6 hours, 2,000 steps, target 8,000 steps. Its recorded attainment is 25%; it is not necessarily a full-day result.

`additional_id` is a candidate record identifier; `sdv_user_id` identifies a participant; group/type fields describe context. `calendar_date`, week number and weekday label the record. Start/end/duration describe its measurement period. `steps` and `steps_goal` are counts, not rates. We do not average identifiers or assume the goal belongs to a full day without documentation.

In [ ]:
display(pd.DataFrame({'column': s.columns, 'pandas_type': s.dtypes.astype(str).values,
    'null_rows': s.isna().sum().values, 'distinct_nonnull': s.nunique().values}))
display(pd.DataFrame([{'exact_duplicate_rows_after_first': int(s.duplicated().sum()),
    'zero_steps': int(s.steps.eq(0).sum()), 'negative_steps': int(s.steps.lt(0).sum()),
    'missing_steps': int(s.steps.isna().sum()), 'zero_goals': int(s.steps_goal.eq(0).sum()),
    'negative_goals': int(s.steps_goal.lt(0).sum()), 'missing_goals': int(s.steps_goal.isna().sum())}]))
display(s[['steps','steps_goal']].describe().T)


## 3. Test the grain and candidate keys
The **grain** means what one row stands for. Test participant/date, participant/period, participant/date/period and the record identifier separately in each table. A usable one-to-one key must have no nulls and no duplicates on either side. We never drop duplicates just to make a join succeed.

In [ ]:
date_key = ['sdv_user_id','calendar_date']
period_key = ['sdv_user_id','start_measure_period_sec','end_measure_period_sec']
strict_key = date_key + ['start_measure_period_sec','end_measure_period_sec']
candidates = {'record_id':['additional_id'], 'participant_date':date_key,
              'participant_period':period_key, 'participant_date_period':strict_key}
def valid_key(df, key):
    return not df[key].isna().any(axis=None) and not df.duplicated(key).any()
audit=[]
for name, df in tables.items():
    for label, key in candidates.items():
        audit.append({'table':name,'key':label,'null_key_rows':int(df[key].isna().any(axis=1).sum()),
            'rows_in_duplicate_keys':int(df.duplicated(key,keep=False).sum()),
            'unique_nonnull_key':valid_key(df,key)})
display(pd.DataFrame(audit))


## 4. Check periods and coverage
Compare duration against end minus start (and against an inclusive end convention). Negative/zero periods are invalid for duration interpretation. A 24-hour period is a duration check, **not proof of continuous wear**. 23/25-hour periods could reflect daylight saving; timezone documentation is needed before attributing them to it. UTC conversion below is only a timestamp plausibility check.

Coverage is observed unique dates divided by dates between each participant's first and last observation. It excludes time outside that span and cannot identify enrolment dates. Missing dates are not zero-step dates.

In [ ]:
s['elapsed_sec'] = s.end_measure_period_sec - s.start_measure_period_sec
s['period_hours'] = s.duration_measure_period_sec / 3600
display(pd.DataFrame([{'nonpositive_reported_duration':int(s.duration_measure_period_sec.le(0).sum()),
 'end_not_after_start':int(s.elapsed_sec.le(0).sum()),
 'duration_equals_end_minus_start':int(s.duration_measure_period_sec.eq(s.elapsed_sec).sum()),
 'duration_equals_end_minus_start_plus_one':int(s.duration_measure_period_sec.eq(s.elapsed_sec+1).sum()),
 'exact_24_hour_rows':int(s.period_hours.eq(24).sum()),
 'under_24_hour_rows':int(s.period_hours.lt(24).sum()),
 'over_24_hour_rows':int(s.period_hours.gt(24).sum())}]))
display(s.groupby('period_hours',dropna=False).size().rename('rows').reset_index())
utc_start = pd.to_datetime(s.start_measure_period_sec,unit='s',utc=True,errors='coerce')
display(pd.DataFrame([{'unparseable_unix_second_starts':int(utc_start.isna().sum()),
 'UTC_start_date_matches_calendar_label':int(utc_start.dt.tz_localize(None).dt.normalize().eq(s.calendar_date).sum())}]))
coverage = s.groupby('sdv_user_id').agg(first=('calendar_date','min'),last=('calendar_date','max'),
    observed_dates=('calendar_date','nunique'),records=('calendar_date','size'))
coverage['span_days']=(coverage['last']-coverage['first']).dt.days+1
coverage['missing_dates_within_span']=coverage.span_days-coverage.observed_dates
coverage['observed_date_percent']=100*coverage.observed_dates/coverage.span_days
display(coverage[['observed_dates','records','span_days','missing_dates_within_span','observed_date_percent']].describe().T)
ordered=s.sort_values(period_key)
previous_max_end=ordered.groupby('sdv_user_id').end_measure_period_sec.transform(lambda x:x.cummax().shift())
print('Rows starting before a preceding period ends:', int(ordered.start_measure_period_sec.lt(previous_max_end).sum()))
print('Boundary equality is not counted as overlap; inclusive/exclusive endpoint semantics need confirmation.')


## 5. Validate cross-table correspondence before comparing values
Use participant, date, start and end together for an exact recorded-period match. An outer join reports unmatched keys on both sides, and pandas `validate='one_to_one'` prevents row multiplication. If this key fails, the comparison is skipped for investigation. Matching periods still do not establish device/source equivalence; compare group, duration and record identifiers separately. Data-type labels can legitimately differ between tables.

In [ ]:
comparisons=[]
join_audit=[]
for other, metric in [('activity','steps'),('goals','steps_goal')]:
    right=tables[other]
    if not (valid_key(s,strict_key) and valid_key(right,strict_key)):
        print(other + ': comparison skipped because the exact-period key is not unique and non-null.')
        continue
    fields=[metric,'sdv_group_id','duration_measure_period_sec','additional_id']
    joined=s[strict_key+fields].merge(right[strict_key+fields],on=strict_key,how='outer',
        suffixes=('_steps_table','_other_table'),indicator=True,validate='one_to_one')
    matched=joined.loc[joined._merge.eq('both')]
    join_audit.append({'other_table':other,'steps_only':int(joined._merge.eq('left_only').sum()),
        'other_only':int(joined._merge.eq('right_only').sum()),'matched_periods':len(matched)})
    for field in fields:
        a=matched[field+'_steps_table']; b=matched[field+'_other_table']
        comparable=a.notna() & b.notna()
        comparisons.append({'other_table':other,'field':field,'matched_periods':len(matched),
            'both_nonnull':int(comparable.sum()),'equal_nonnull':int((a.eq(b)&comparable).sum()),
            'different_nonnull':int((a.ne(b)&comparable).sum()),'missing_either':int((~comparable).sum())})
display(pd.DataFrame(join_audit))
display(pd.DataFrame(comparisons))


## 6. Recorded goal attainment with explicit denominators
A row is eligible when steps are present and nonnegative and the goal is present and strictly positive. Zero/negative goals are excluded, not silently treated as success; division by zero is never performed. Report meeting the goal as steps ≥ goal. The denominator is eligible **records**, not all calendar days and not participants. Repeated participants contribute unequal numbers of records.

The 24-hour subset is a sensitivity check; neither it nor the complete set proves full wear. We use the goal in the steps table, with agreement against `goals` assessed above. Disagreement would require clarification before selecting an authoritative source.

In [ ]:
eligible=s.steps.notna() & s.steps.ge(0) & s.steps_goal.notna() & s.steps_goal.gt(0)
s['attainment_percent']=np.nan
s.loc[eligible,'attainment_percent']=100*s.loc[eligible,'steps']/s.loc[eligible,'steps_goal']
attainment=[]
for label, scope in [('all recorded periods',pd.Series(True,index=s.index)),
                     ('reported duration exactly 24 hours',s.period_hours.eq(24))]:
    usable=scope & eligible
    met=usable & s.steps.ge(s.steps_goal)
    attainment.append({'scope':label,'records':int(scope.sum()),'eligible_records':int(usable.sum()),
        'excluded_records':int((scope & ~eligible).sum()),'records_meeting_goal':int(met.sum()),
        'percent_of_eligible_meeting_goal':100*met.sum()/usable.sum() if usable.any() else np.nan})
display(pd.DataFrame(attainment))
display(s.loc[eligible,'attainment_percent'].describe().to_frame())
participant_rates=s.loc[eligible].assign(met=lambda x:x.steps.ge(x.steps_goal)).groupby('sdv_user_id').met.mean()*100
display(participant_rates.describe().to_frame('participant_percent_records_meeting_goal'))
fig, axes=plt.subplots(1,2,figsize=(11,4))
s.steps.dropna().plot.hist(bins=25,ax=axes[0],title='Steps per recorded period')
axes[0].set_xlabel('Recorded steps'); axes[0].set_ylabel('Records')
s.loc[eligible,'attainment_percent'].plot.hist(bins=25,ax=axes[1],title='Recorded goal attainment')
axes[1].axvline(100,color='red',linestyle='--',label='Goal met')
axes[1].set_xlabel('Steps / positive goal (%)'); axes[1].set_ylabel('Eligible records'); axes[1].legend()
plt.tight_layout(); plt.show()


## Verified findings from this run
- There are **987 steps records across 8 participants**. All tested candidate keys are unique and non-null in each of the three tables.
- Exact participant/date/start/end joins match **all 987 records** in each comparison, with no unmatched records. Steps agree with activity; steps goals agree with goals; compared record identifiers, groups and durations also agree.
- **361 / 987 eligible recorded periods (36.58%)** meet the positive recorded goal. For reported durations of exactly 24 hours, the result is **355 / 970 (36.60%)**. These are record-weighted descriptive percentages.
- **970** records report 24 hours, **16** report less, and **1** reports more. All reported durations equal end minus start. **Two** records begin before a preceding period ends, so summing periods into total follow-up or total steps could double-count overlapping time.
- No UTC start date matches the calendar-date label under the Unix-seconds interpretation. This does not prove an error: local dates may differ from UTC. Confirm timezone and timestamp conventions before converting dates or attributing 23/25-hour windows to daylight saving.
- Observed-date coverage within individual first-to-last spans ranges from **89.36% to 100%**; observed dates range from **16 to 250 per participant**. Coverage does not measure wear time or observation outside these spans.

**A sentence for the presentation:** “After validating the record keys and matching measurement periods across three tables, 36.58% of eligible recorded periods met their recorded steps goal; incomplete periods, overlap, timezone conventions and unequal follow-up limit the interpretation.”


## 7. Presentation notes and limits
Explain the analysis in this order:

1. **Row meaning:** a participant's recorded measurement period labelled by a date; confirm the source definition before calling it a complete day.
2. **Trust checks:** schema, nulls, duplicate rows, candidate keys, duration consistency and observed-date coverage.
3. **Table comparison:** exact-period keys and explicit one-to-one joins establish which records match; matched counts and field agreement provide the evidence.
4. **Main result:** read the eligible denominator, goal-meeting count and percentage from section 6. Say “recorded periods meeting a positive recorded goal.”
5. **Limitations:** incomplete wear cannot be determined from these summary tables; missing dates are not inactivity; participants have unequal follow-up; eight participants cannot represent a population. No causal conclusion follows.

**Questions for the data owner:** Are timestamps Unix seconds and what timezone defines `calendar_date`? Are ends inclusive? Does duration represent a reporting window or actual wear time? Are targets daily/adaptive? Which table is authoritative if values differ? What defines enrolment and expected observation days?

No CSV export, RAG, or predictive model is included. Existing notebooks are unchanged.